In [ ]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(repr)
options(repr.plot.width=10, repr.plot.height=8)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("all", "old", "young")
colorSamples <- c("old"="#A58065",
                "young"="#7FCFF2", 
                "all"="#92A8AC")
for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

thrMinCells <- 500 * 0.05

In [ ]:
load("workspaces/mm_RA_simulation_evaluation_objectCreation.Rdata")

# Evaluation

## Upset plots of detected TEs and TPs

### Upsets of detected TEs

In [ ]:
# create df with features family info for each object; list of samples, containing list of tools
familyDf_list <- list()

# create list of subsetted objects containing only elements of one family
# nested list of samples, tools and orders
objList_order <- list()
splatter_objs_order <- list()

for (sample in samples){
    print(sample)
    familyDf_list[[sample]] <- list()
    objList_order[[sample]] <- list()
    splatter_objs_order[[sample]] <- list()
    
    TPlist_byOrder <- list()
    detectedList_byOrder <- list()
    
    # PRE-POPULATE: Process simulated data first to get all orders
    obj_sim <- splatter_objs[[sample]]
    featuresOrder_sim <- conversion_table$class[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresFamily_sim <- conversion_table$family[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresSubfamily_sim <- conversion_table$subfamily[match(Features(obj_sim), conversion_table$stellarscopeID)]
    
    familyDf_sim <- as.data.frame(cbind(Features(obj_sim), featuresOrder_sim, featuresFamily_sim, featuresSubfamily_sim))
    familyDf_list[[sample]][["simulated"]] <- familyDf_sim
    
    # Create all order subsets for simulated data
    all_orders <- unique(familyDf_sim$featuresOrder_sim)
    for(order in all_orders){
        obj_sub_sim <- obj_sim[Features(obj_sim) %in% familyDf_sim$V1[familyDf_sim$featuresOrder_sim == order],]
        splatter_objs_order[[sample]][[order]] <- obj_sub_sim
    }
    
    # Initialize detectedList for each order
    for(order in all_orders){
        detectedList_byOrder[[order]] <- list()
        detectedList_byOrder[[order]][["simulated"]] <- Features(splatter_objs_order[[sample]][[order]])
    }
    
    # NOW process other tools
    for (tool in names(objList)){
        print(tool)
        objList_order[[sample]][[tool]] <- list()
        
        obj <- objList[[tool]][[sample]]
        
        featuresOrder <- conversion_table$class[match(Features(obj), conversion_table$stellarscopeID)]
        featuresFamily <- conversion_table$family[match(Features(obj), conversion_table$stellarscopeID)]
        featuresSubfamily <- conversion_table$subfamily[match(Features(obj), conversion_table$stellarscopeID)]
        
        familyDf <- as.data.frame(cbind(Features(obj), featuresOrder, featuresFamily, featuresSubfamily))
        familyDf_list[[sample]][[tool]] <- familyDf
        
        TPlist_byOrder[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- list()
        
        for(order in unique(familyDf$featuresOrder)){
            print(order)
            
            obj_sub <- obj[Features(obj) %in% familyDf$V1[familyDf$featuresOrder == order],]
            print(obj_sub)
            
            objList_order[[sample]][[tool]][[order]] <- obj_sub
            
            # Only add to detectedList if this order exists in simulated data
            if(order %in% all_orders){
                detectedList_byOrder[[order]][[tool]] <- Features(obj_sub)
                
                # Calculate TP for this order
                TPlist_byOrder[[sub("^(.*)_.*$", "\\1", obj@project.name)]][[order]] <- intersect(
                    Features(splatter_objs_order[[sample]][[order]]), 
                    Features(obj_sub))
            }
        }
    } # Close tool loop
    
    # Create upset plot for each order
    for(order in all_orders){
        print(paste0("Creating upset plot for order: ", order))
        
        detectedList_order <- detectedList_byOrder[[order]]
        
        # Only plot if there are features detected
        if(length(detectedList_order) > 0 && sum(lengths(detectedList_order)) > 0){
            
            #pdf(file=paste0("figures_",dataset_id,"_",sample,"/upset_detected_", order, ".pdf"), width = 13, height = 7.5)
            
            show(UpSetR::upset(fromList(detectedList_order), 
                    nintersects = 30,
                    sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
                    text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
                    order.by=c("freq"),
                    point.size=3, mb.ratio = c(0.5, 0.5),
                    set_size.show=F, set_size.scale_max= max(lengths(detectedList_order))+1000,
                    nsets=length(detectedList_order),
                    sets.x.label="N. detected loci",
                    mainbar.y.label="Intersection size")
            )
            grid.text(paste0("Detected TEs - ", sample, ", ", order), x = 0.65, y=0.95, gp=gpar(fontsize=18))
            
            #dev.off()
        }
    }
    
} # Close sample loop


## Total TE counts per cell

## Precision / Recall by Order


In [ ]:
precisionTool_byOrder <- list()
recallTool_byOrder <- list()
f1Tool_byOrder <- list()

for (sample in samples){
    layer <- "counts"
    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)
    
    precisionTool_byOrder[[sample]] <- list()
    recallTool_byOrder[[sample]] <- list()
    f1Tool_byOrder[[sample]] <- list()
    
    all_orders <- unique(familyDf_list[[sample]][["simulated"]]$featuresOrder_sim)
    
    for(tool in names(objList)){
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
        
        precisionTool_byOrder[[sample]][[tool]] <- list()
        recallTool_byOrder[[sample]][[tool]] <- list()
        f1Tool_byOrder[[sample]][[tool]] <- list()
        
        for(order in all_orders){
            # Get features for this order
            features_splatter_order <- Features(splatter_objs_order[[sample]][[order]])
            
            if(!is.null(objList_order[[sample]][[tool]][[order]])){
                features_tool_order <- Features(objList_order[[sample]][[tool]][[order]])
            } else {
                features_tool_order <- character(0)
            }
            
            # Aggregate across ALL cells
            nTP <- 0
            nFP <- 0
            nFN <- 0
            
            for(cell in Cells(splatter_objs[[sample]])){
                exprSplatter <- names(matSplatter[features_splatter_order, cell])[matSplatter[features_splatter_order, cell] > 0]
                exprTool <- names(matTool[features_tool_order, cell])[matTool[features_tool_order, cell] > 0]
                
                TP <- intersect(exprSplatter, exprTool)
                FP <- setdiff(exprTool, exprSplatter)
                FN <- setdiff(exprSplatter, exprTool)
                
                nTP <- nTP + length(TP)
                nFP <- nFP + length(FP)
                nFN <- nFN + length(FN)
            }
            
            # Compute metrics across all cells
            precision <- nTP / (nTP + nFP)
            recall <- nTP / (nTP + nFN)
            
            # F1 score: harmonic mean of precision and recall
            # Handle edge cases where precision or recall is 0 or NA
            if(is.na(precision) || is.na(recall) || (precision == 0 && recall == 0)){
                f1 <- 0
            } else {
                f1 <- 2 * (precision * recall) / (precision + recall)
            }
            
            precisionTool_byOrder[[sample]][[tool]][[order]] <- precision
            recallTool_byOrder[[sample]][[tool]][[order]] <- recall
            f1Tool_byOrder[[sample]][[tool]][[order]] <- f1
        }
    }
}

In [ ]:
options(repr.plot.width=14, repr.plot.height=14)

library(ggplot2)
library(tidyr)
library(dplyr)

# Prepare data for plotting
plot_data_list <- list()

for (sample in samples){
    for(tool in names(objList)){
        if(!is.null(precisionTool_byOrder[[sample]][[tool]])){
            for(order in names(precisionTool_byOrder[[sample]][[tool]])){
                
                precision <- precisionTool_byOrder[[sample]][[tool]][[order]]
                recall <- recallTool_byOrder[[sample]][[tool]][[order]]
                f1 <- f1Tool_byOrder[[sample]][[tool]][[order]]
                
                plot_data_list[[length(plot_data_list) + 1]] <- data.frame(
                    sample = sample,
                    tool = tool,
                    order = order,
                    precision = precision,
                    recall = recall,
                    f1 = f1
                )
            }
        }
    }
}

plot_data <- bind_rows(plot_data_list)

# Set factor levels for tool based on colorTools order
plot_data$tool <- factor(plot_data$tool, levels = names(colorTools))

# Reshape data to long format
plot_data_long <- plot_data %>%
    pivot_longer(cols = c(precision, recall, f1), 
                 names_to = "metric", 
                 values_to = "value")

# Set metric order
plot_data_long$metric <- factor(plot_data_long$metric, 
                                levels = c("precision", "recall", "f1"),
                                labels = c("Precision", "Recall", "F1 Score"))

# Bar plot with orders on y-axis, values on x-axis
p_flipped <- ggplot(plot_data_long, aes(x = value, y = order, fill = tool)) +
    geom_bar(stat = "identity", position = "dodge") +
    facet_grid(sample ~ metric, scales = "free_y", space = "free_y") +
    scale_fill_manual(values = colorTools, drop = FALSE) +
    theme_pubclean() +
    theme(text = element_text(size = 15),
          strip.text = element_text(size = 15)) +
    labs(title = "Performance Metrics by TE Order",
         x = "Score",
         y = "TE Order",
         fill = "Tool") +
    xlim(0, 1)

print(p_flipped)

# Save plot
ggsave(paste0("figures_", dataset_id, "/metrics_flipped_by order.pdf"), p_flipped, width = 14, height = 14)

## By Family

In [ ]:
precisionTool_byFamily <- list()
recallTool_byFamily <- list()
f1Tool_byFamily <- list()

# First, create family-based subsets similar to order-based subsets
familyDf_list <- list()
objList_family <- list()
splatter_objs_family <- list()

for (sample in samples){
    
    print(sample)
    familyDf_list[[sample]] <- list()
    objList_family[[sample]] <- list()
    splatter_objs_family[[sample]] <- list()
    
    # PRE-POPULATE: Process simulated data first to get all families
    obj_sim <- splatter_objs[[sample]]
    featuresOrder_sim <- conversion_table$class[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresFamily_sim <- conversion_table$family[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresSubfamily_sim <- conversion_table$subfamily[match(Features(obj_sim), conversion_table$stellarscopeID)]
    
    familyDf_sim <- as.data.frame(cbind(Features(obj_sim), featuresOrder_sim, featuresFamily_sim, featuresSubfamily_sim))
    familyDf_list[[sample]][["simulated"]] <- familyDf_sim
    
    # Create all family subsets for simulated data
    all_families <- unique(familyDf_sim$featuresFamily_sim)
    all_families <- all_families[!is.na(all_families)]  # Remove NA families
    
    for(family in all_families){
        obj_sub_sim <- obj_sim[Features(obj_sim) %in% familyDf_sim$V1[familyDf_sim$featuresFamily_sim == family],]
        splatter_objs_family[[sample]][[family]] <- obj_sub_sim
    }
    
    # NOW process other tools
    for (tool in names(objList)){
        print(tool)
        objList_family[[sample]][[tool]] <- list()
        
        obj <- objList[[tool]][[sample]]
        
        featuresOrder <- conversion_table$class[match(Features(obj), conversion_table$stellarscopeID)]
        featuresFamily <- conversion_table$family[match(Features(obj), conversion_table$stellarscopeID)]
        featuresSubfamily <- conversion_table$subfamily[match(Features(obj), conversion_table$stellarscopeID)]
        
        familyDf <- as.data.frame(cbind(Features(obj), featuresOrder, featuresFamily, featuresSubfamily))
        familyDf_list[[sample]][[tool]] <- familyDf
        
        for(family in unique(familyDf$featuresFamily)){
            if(!is.na(family)){
                print(family)
                
                obj_sub <- obj[Features(obj) %in% familyDf$V1[familyDf$featuresFamily == family],]
                
                objList_family[[sample]][[tool]][[family]] <- obj_sub
            }
        }
    }
}

# Now calculate precision, recall, and F1 by family
precisionTool_byFamily <- list()
recallTool_byFamily <- list()
f1Tool_byFamily <- list()

for (sample in samples){
    layer <- "counts"
    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)
    
    precisionTool_byFamily[[sample]] <- list()
    recallTool_byFamily[[sample]] <- list()
    f1Tool_byFamily[[sample]] <- list()
    
    all_families <- unique(familyDf_list[[sample]][["simulated"]]$featuresFamily_sim)
    all_families <- all_families[!is.na(all_families)]  # Remove NA families
    
    for(tool in names(objList)){
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
        
        precisionTool_byFamily[[sample]][[tool]] <- list()
        recallTool_byFamily[[sample]][[tool]] <- list()
        f1Tool_byFamily[[sample]][[tool]] <- list()
        
        for(family in all_families){
            # Get features for this family
            features_splatter_family <- Features(splatter_objs_family[[sample]][[family]])
            
            if(!is.null(objList_family[[sample]][[tool]][[family]])){
                features_tool_family <- Features(objList_family[[sample]][[tool]][[family]])
            } else {
                features_tool_family <- character(0)
            }
            
            # Aggregate across ALL cells
            nTP <- 0
            nFP <- 0
            nFN <- 0
            
            for(cell in Cells(splatter_objs[[sample]])){
                exprSplatter <- names(matSplatter[features_splatter_family, cell])[matSplatter[features_splatter_family, cell] > 0]
                exprTool <- names(matTool[features_tool_family, cell])[matTool[features_tool_family, cell] > 0]
                
                TP <- intersect(exprSplatter, exprTool)
                FP <- setdiff(exprTool, exprSplatter)
                FN <- setdiff(exprSplatter, exprTool)
                
                nTP <- nTP + length(TP)
                nFP <- nFP + length(FP)
                nFN <- nFN + length(FN)
            }
            
            # Compute metrics across all cells
            precision <- nTP / (nTP + nFP)
            recall <- nTP / (nTP + nFN)
            
            # F1 score: harmonic mean of precision and recall
            # Handle edge cases where precision or recall is 0 or NA
            if(is.na(precision) || is.na(recall) || (precision == 0 && recall == 0)){
                f1 <- 0
            } else {
                f1 <- 2 * (precision * recall) / (precision + recall)
            }
            
            precisionTool_byFamily[[sample]][[tool]][[family]] <- precision
            recallTool_byFamily[[sample]][[tool]][[family]] <- recall
            f1Tool_byFamily[[sample]][[tool]][[family]] <- f1
        }
    }
}

# Plot by family
library(ggplot2)
library(tidyr)
library(dplyr)

# Prepare data for plotting
plot_data_list <- list()

for (sample in samples){
    for(tool in names(objList)){
        if(!is.null(precisionTool_byFamily[[sample]][[tool]])){
            for(family in names(precisionTool_byFamily[[sample]][[tool]])){
                
                precision <- precisionTool_byFamily[[sample]][[tool]][[family]]
                recall <- recallTool_byFamily[[sample]][[tool]][[family]]
                f1 <- f1Tool_byFamily[[sample]][[tool]][[family]]
                
                plot_data_list[[length(plot_data_list) + 1]] <- data.frame(
                    sample = sample,
                    tool = tool,
                    family = family,
                    precision = precision,
                    recall = recall,
                    f1 = f1
                )
            }
        }
    }
}

plot_data <- bind_rows(plot_data_list)

# Set factor levels for tool based on colorTools order
plot_data$tool <- factor(plot_data$tool, levels = names(colorTools))

# Reshape data to long format
plot_data_long <- plot_data %>%
    pivot_longer(cols = c(precision, recall, f1), 
                 names_to = "metric", 
                 values_to = "value")

# Set metric order
plot_data_long$metric <- factor(plot_data_long$metric, 
                                levels = c("precision", "recall", "f1"),
                                labels = c("Precision", "Recall", "F1 Score"))

# Bar plot with families on y-axis, values on x-axis
p_family_flipped <- ggplot(plot_data_long, aes(x = value, y = family, fill = tool)) +
    geom_bar(stat = "identity", position = "dodge") +
    facet_grid(sample ~ metric, scales = "free_y", space = "free_y") +
    scale_fill_manual(values = colorTools, drop = FALSE) +
    theme_bw() +
    theme(text = element_text(size = 12),
          strip.text = element_text(size = 11)) +
    labs(title = "Performance Metrics by TE Family",
         x = "Score",
         y = "TE Family",
         fill = "Tool") +
    xlim(0, 1)

print(p_family_flipped)

# ggsave(paste0("figures_", dataset_id, "/metrics_by_family.pdf"), p_family_flipped, width = 14, height = 10)

In [ ]:
options(repr.plot.width=12, repr.plot.height=8)

# Bar plot with families on y-axis, values on x-axis
p_family_flipped <- ggplot(plot_data_long[plot_data_long$sample=="young",], 
        aes(x = value, y = family, fill = tool)) +
    geom_bar(stat = "identity", position = "dodge") +
    facet_grid(sample ~ metric, scales = "free_y", space = "free_y") +
    scale_fill_manual(values = colorTools, drop = FALSE) +
    theme_pubclean() +
    theme(text = element_text(size = 15),
          strip.text = element_text(size = 15)) +
    labs(title = "Performance Metrics by TE Family",
         x = "Score",
         y = "TE Family",
         fill = "Tool") +
    xlim(0, 1)

print(p_family_flipped)

# Save plot
ggsave(paste0("figures_", dataset_id, "_young/metrics_flipped_by family.pdf"), width = 12, height = 8)

In [ ]:
options(repr.plot.width=13, repr.plot.height=8)

plot_data_long_young <- plot_data_long[plot_data_long$sample=="young",]

family_order <- plot_data_long_young %>%
    filter(tool == "Stellarscope") %>%
    filter(metric == "F1 Score") %>%
    arrange(desc(value)) %>%
    pull(family)
# Set family as factor with custom order
plot_data_long_young$family <- factor(plot_data_long_young$family, levels = family_order)

# Bar plot with families on y-axis, values on x-axis
p_family_flipped <- ggplot(plot_data_long_young, 
        aes(x = value, y = family, fill = tool)) +
    geom_bar(stat = "identity", position = "dodge", width=0.7) +
    facet_grid(sample ~ metric, scales = "free_y", space = "free_y") +
    scale_fill_manual(values = colorTools, drop = FALSE) +
    theme_minimal() +
    theme(text = element_text(size = 18),
          strip.text = element_text(size = 18),
          panel.spacing.x = unit(4, "lines"),  # space between columns
          panel.spacing.y = unit(1.2, "lines")   # space between rows (optional)
) +
    labs(title = "Performance Metrics by TE Family",
         x = "Score",
         y = "TE Family",
         fill = "Tool") +
    xlim(0, 1)

p_family_flipped

# Save plot
ggsave(paste0("figures_", dataset_id, "_young/metrics_flipped_by_family_minimal.pdf"), width = 13, height = 8)

In [ ]:
# Compute precision and recall by family

precisionTool_byFamily <- list()
recallTool_byFamily <- list()
f1Tool_byFamily <- list()

# First, create family-based subsets similar to order-based subsets
familyDf_list <- list()
objList_family <- list()
splatter_objs_family <- list()

for (sample in samples){
    
    print(sample)

    familyDf_list[[sample]] <- list()
    objList_family[[sample]] <- list()
    splatter_objs_family[[sample]] <- list()
    
    # Process simulated data first.q..0
    obj_sim <- splatter_objs[[sample]]
    featuresOrder_sim <- conversion_table$class[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresFamily_sim <- conversion_table$family[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresSubfamily_sim <- conversion_table$subfamily[match(Features(obj_sim), conversion_table$stellarscopeID)]
    
    familyDf_sim <- as.data.frame(cbind(Features(obj_sim), featuresOrder_sim, featuresFamily_sim, featuresSubfamily_sim))
    familyDf_list[[sample]][["simulated"]] <- familyDf_sim
    
    # Create all family subsets for simulated data
    all_families <- unique(familyDf_sim$featuresFamily_sim)
    all_families <- all_families[!is.na(all_families)]  # Remove NA families
    
    for(family in all_families){
        obj_sub_sim <- obj_sim[Features(obj_sim) %in% familyDf_sim$V1[familyDf_sim$featuresFamily_sim == family],]
        splatter_objs_family[[sample]][[family]] <- obj_sub_sim
    }
    
    # NOW process other tools
    for (tool in names(objList)){
        print(tool)
        objList_family[[sample]][[tool]] <- list()
        
        obj <- objList[[tool]][[sample]]
        
        featuresOrder <- conversion_table$class[match(Features(obj), conversion_table$stellarscopeID)]
        featuresFamily <- conversion_table$family[match(Features(obj), conversion_table$stellarscopeID)]
        featuresSubfamily <- conversion_table$subfamily[match(Features(obj), conversion_table$stellarscopeID)]
        
        familyDf <- as.data.frame(cbind(Features(obj), featuresOrder, featuresFamily, featuresSubfamily))
        familyDf_list[[sample]][[tool]] <- familyDf
        
        for(family in unique(familyDf$featuresFamily)){
            if(!is.na(family)){
                print(family)
                
                obj_sub <- obj[Features(obj) %in% familyDf$V1[familyDf$featuresFamily == family],]
                
                objList_family[[sample]][[tool]][[family]] <- obj_sub
            }
        }
    }
}

# Now calculate precision, recall, and F1 by family
precisionTool_byFamily <- list()
recallTool_byFamily <- list()
f1Tool_byFamily <- list()

for (sample in samples){
    layer <- "counts"
    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)
    
    precisionTool_byFamily[[sample]] <- list()
    recallTool_byFamily[[sample]] <- list()
    f1Tool_byFamily[[sample]] <- list()
    
    all_families <- unique(familyDf_list[[sample]][["simulated"]]$featuresFamily_sim)
    all_families <- all_families[!is.na(all_families)]  # Remove NA families
    
    for(tool in names(objList)){
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
        
        precisionTool_byFamily[[sample]][[tool]] <- list()
        recallTool_byFamily[[sample]][[tool]] <- list()
        f1Tool_byFamily[[sample]][[tool]] <- list()
        
        for(family in all_families){
            # Get features for this family
            features_splatter_family <- Features(splatter_objs_family[[sample]][[family]])
            
            if(!is.null(objList_family[[sample]][[tool]][[family]])){
                features_tool_family <- Features(objList_family[[sample]][[tool]][[family]])
            } else {
                features_tool_family <- character(0)
            }
            
            # Aggregate across ALL cells
            nTP <- 0
            nFP <- 0
            nFN <- 0
            
            for(cell in Cells(splatter_objs[[sample]])){
                exprSplatter <- names(matSplatter[features_splatter_family, cell])[matSplatter[features_splatter_family, cell] > 0]
                exprTool <- names(matTool[features_tool_family, cell])[matTool[features_tool_family, cell] > 0]
                
                TP <- intersect(exprSplatter, exprTool)
                FP <- setdiff(exprTool, exprSplatter)
                FN <- setdiff(exprSplatter, exprTool)
                
                nTP <- nTP + length(TP)
                nFP <- nFP + length(FP)
                nFN <- nFN + length(FN)
            }
            
            # Compute metrics across all cells
            precision <- nTP / (nTP + nFP)
            recall <- nTP / (nTP + nFN)
            
            # F1 score: harmonic mean of precision and recall
            # Handle edge cases where precision or recall is 0 or NA
            if(is.na(precision) || is.na(recall) || (precision == 0 && recall == 0)){
                f1 <- 0
            } else {
                f1 <- 2 * (precision * recall) / (precision + recall)
            }
            
            precisionTool_byFamily[[sample]][[tool]][[family]] <- precision
            recallTool_byFamily[[sample]][[tool]][[family]] <- recall
            f1Tool_byFamily[[sample]][[tool]][[family]] <- f1
        }
    }
}

# Plot by family
library(ggplot2)
library(tidyr)
library(dplyr)

# Prepare data for plotting
plot_data_list <- list()

for (sample in samples){
    for(tool in names(objList)){
        if(!is.null(precisionTool_byFamily[[sample]][[tool]])){
            for(family in names(precisionTool_byFamily[[sample]][[tool]])){
                
                precision <- precisionTool_byFamily[[sample]][[tool]][[family]]
                recall <- recallTool_byFamily[[sample]][[tool]][[family]]
                f1 <- f1Tool_byFamily[[sample]][[tool]][[family]]
                
                plot_data_list[[length(plot_data_list) + 1]] <- data.frame(
                    sample = sample,
                    tool = tool,
                    family = family,
                    precision = precision,
                    recall = recall,
                    f1 = f1
                )
            }
        }
    }
}

plot_data <- bind_rows(plot_data_list)

# Set factor levels for tool based on colorTools order
plot_data$tool <- factor(plot_data$tool, levels = names(colorTools))

# Reshape data to long format
plot_data_long <- plot_data %>%
    pivot_longer(cols = c(precision, recall, f1), 
                 names_to = "metric", 
                 values_to = "value")

# Set metric order
plot_data_long$metric <- factor(plot_data_long$metric, 
                                levels = c("precision", "recall", "f1"),
                                labels = c("Precision", "Recall", "F1 Score"))

# Bar plot with families on y-axis, values on x-axis
p_family_flipped <- ggplot(plot_data_long, aes(x = value, y = family, fill = tool)) +
    geom_bar(stat = "identity", position = "dodge") +
    facet_grid(sample ~ metric, scales = "free_y", space = "free_y") +
    scale_fill_manual(values = colorTools, drop = FALSE) +
    theme_bw() +
    theme(text = element_text(size = 12),
          strip.text = element_text(size = 11)) +
    labs(title = "Performance Metrics by TE Family",
         x = "Score",
         y = "TE Family",
         fill = "Tool") +
    xlim(0, 1)

print(p_family_flipped)

# ggsave(paste0("figures_", dataset_id, "/metrics_by_family.pdf"), p_family_flipped, width = 14, height = 10)

## By Subfamily

In [ ]:
# Compute precision and recall by family

precisionTool_byFamily <- list()
recallTool_byFamily <- list()
f1Tool_byFamily <- list()

# First, create subfamily-based subsets similar to order-based subsets
subfamilyDf_list <- list()
objList_subfamily <- list()
splatter_objs_subfamily <- list()

for (sample in samples){
    
    print(sample)

    subfamilyDf_list[[sample]] <- list()
    objList_subfamily[[sample]] <- list()
    splatter_objs_subfamily[[sample]] <- list()
    
    # Process simulated data first.q..0
    obj_sim <- splatter_objs[[sample]]
    featuresOrder_sim <- conversion_table$class[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresFamily_sim <- conversion_table$family[match(Features(obj_sim), conversion_table$stellarscopeID)]
    featuresSubfamily_sim <- conversion_table$subfamily[match(Features(obj_sim), conversion_table$stellarscopeID)]
    
    subfamilyDf_sim <- as.data.frame(cbind(Features(obj_sim), featuresOrder_sim, featuresFamily_sim, featuresSubfamily_sim))
    subfamilyDf_list[[sample]][["simulated"]] <- subfamilyDf_sim
    
    # Create all subfamily subsets for simulated data
    all_subfamilies <- unique(subfamilyDf_sim$featuresSubfamily_sim)
    all_subfamilies <- all_subfamilies[!is.na(all_subfamilies)]  # Remove NA families
    
    for(subfamily in all_subfamilies){
        obj_sub_sim <- obj_sim[Features(obj_sim) %in% subfamilyDf_sim$V1[subfamilyDf_sim$featuresSubamily_sim == subfamily],]
        splatter_objs_subfamily[[sample]][[subfamily]] <- obj_sub_sim
    }
    
    # NOW process other tools
    for (tool in names(objList)){
        print(tool)
        objList_subfamily[[sample]][[tool]] <- list()
        
        obj <- objList[[tool]][[sample]]
        
        featuresOrder <- conversion_table$class[match(Features(obj), conversion_table$stellarscopeID)]
        featuresFamily <- conversion_table$family[match(Features(obj), conversion_table$stellarscopeID)]
        featuresSubfamily <- conversion_table$subfamily[match(Features(obj), conversion_table$stellarscopeID)]
        
        familyDf <- as.data.frame(cbind(Features(obj), featuresOrder, featuresFamily, featuresSubfamily))
        subfamilyDf_list[[sample]][[tool]] <- familyDf
        
        for(subfamily in unique(familyDf$featuresSubfamily)){
            if(!is.na(subfamily)){
                print(subfamily)
                
                obj_sub <- obj[Features(obj) %in% familyDf$V1[familyDf$featuresSubamily == subfamily],]
                
                objList_subfamily[[sample]][[tool]][[subfamily]] <- obj_sub
            }
        }
    }
}

# Now calculate precision, recall, and F1 by subfamily
precisionTool_bySubFamily <- list()
recallTool_bySubFamily <- list()
f1Tool_bySubFamily <- list()

for (sample in samples){
    layer <- "counts"
    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)
    
    precisionTool_bySubFamily[[sample]] <- list()
    recallTool_bySubFamily[[sample]] <- list()
    f1Tool_bySubFamily[[sample]] <- list()
    
    all_families <- unique(subfamilyDf_list[[sample]][["simulated"]]$featuresSubfamily_sim)
    all_families <- all_families[!is.na(all_families)]  # Remove NA families
    
    for(tool in names(objList)){
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
        
        precisionTool_bySubFamily[[sample]][[tool]] <- list()
        recallTool_bySubFamily[[sample]][[tool]] <- list()
        f1Tool_bySubFamily[[sample]][[tool]] <- list()
        
        for(subfamily in all_families){
            # Get features for this subfamily
            features_splatter_subfamily <- Features(splatter_objs_subfamily[[sample]][[subfamily]])
            
            if(!is.null(objList_subfamily[[sample]][[tool]][[subfamily]])){
                features_tool_subfamily <- Features(objList_subfamily[[sample]][[tool]][[subfamily]])
            } else {
                features_tool_subfamily <- character(0)
            }
            
            # Aggregate across ALL cells
            nTP <- 0
            nFP <- 0
            nFN <- 0
            
            for(cell in Cells(splatter_objs[[sample]])){
                exprSplatter <- names(matSplatter[features_splatter_subfamily, cell])[matSplatter[features_splatter_subfamily, cell] > 0]
                exprTool <- names(matTool[features_tool_subfamily, cell])[matTool[features_tool_subfamily, cell] > 0]
                
                TP <- intersect(exprSplatter, exprTool)
                FP <- setdiff(exprTool, exprSplatter)
                FN <- setdiff(exprSplatter, exprTool)
                
                nTP <- nTP + length(TP)
                nFP <- nFP + length(FP)
                nFN <- nFN + length(FN)
            }
            
            # Compute metrics across all cells
            precision <- nTP / (nTP + nFP)
            recall <- nTP / (nTP + nFN)
            
            # F1 score: harmonic mean of precision and recall
            # Handle edge cases where precision or recall is 0 or NA
            if(is.na(precision) || is.na(recall) || (precision == 0 && recall == 0)){
                f1 <- 0
            } else {
                f1 <- 2 * (precision * recall) / (precision + recall)
            }
            
            precisionTool_bySubFamily[[sample]][[tool]][[subfamily]] <- precision
            recallTool_bySubFamily[[sample]][[tool]][[subfamily]] <- recall
            f1Tool_bySubFamily[[sample]][[tool]][[subfamily]] <- f1
        }
    }
}

# Plot by subfamily
library(ggplot2)
library(tidyr)
library(dplyr)

# Prepare data for plotting
plot_data_list <- list()

for (sample in samples){
    for(tool in names(objList)){
        if(!is.null(precisionTool_bySubFamily[[sample]][[tool]])){
            for(subfamily in names(precisionTool_bySubFamily[[sample]][[tool]])){
                
                precision <- precisionTool_bySubFamily[[sample]][[tool]][[subfamily]]
                recall <- recallTool_bySubFamily[[sample]][[tool]][[subfamily]]
                f1 <- f1Tool_bySubFamily[[sample]][[tool]][[subfamily]]
                
                plot_data_list[[length(plot_data_list) + 1]] <- data.frame(
                    sample = sample,
                    tool = tool,
                    subfamily = subfamily,
                    precision = precision,
                    recall = recall,
                    f1 = f1
                )
            }
        }
    }
}

plot_data <- bind_rows(plot_data_list)

# Set factor levels for tool based on colorTools order
plot_data$tool <- factor(plot_data$tool, levels = names(colorTools))

# Reshape data to long format
plot_data_long <- plot_data %>%
    pivot_longer(cols = c(precision, recall, f1), 
                 names_to = "metric", 
                 values_to = "value")

# Set metric order
plot_data_long$metric <- factor(plot_data_long$metric, 
                                levels = c("precision", "recall", "f1"),
                                labels = c("Precision", "Recall", "F1 Score"))

# Bar plot with families on y-axis, values on x-axis
p_subfamily_flipped <- ggplot(plot_data_long, aes(x = value, y = subfamily, fill = tool)) +
    geom_bar(stat = "identity", position = "dodge") +
    facet_grid(sample ~ metric, scales = "free_y", space = "free_y") +
    scale_fill_manual(values = colorTools, drop = FALSE) +
    theme_bw() +
    theme(text = element_text(size = 12),
          strip.text = element_text(size = 11)) +
    labs(title = "Performance Metrics by TE Subfamily",
         x = "Score",
         y = "TE Subfamily",
         fill = "Tool") +
    xlim(0, 1)

print(p_subfamily_flipped)

ggsave(paste0("figures_", dataset_id, "/metrics_by_subfamily.pdf"), p_family_flipped, width = 14, height = 10)